# Physics-trained network (Sun et al., 2025)

A U-Net trained without ground truth: it maps the upward-continued data $y$ to a ground-level estimate
$\hat x$ and is trained only on the physics loss $\| A\hat x - y \|_1$, where $A$ is the 800 m continuation.
It has no training set: it is fitted directly on the 10 test observations, and the ground truth is used
only for evaluation. Weights: `weights/physics_trained.pth`.

**How the weights were obtained.** `scripts/train_physics_trained.py` fits a freshly initialised network to the
10 observations at once (one batch, so one epoch is one update): Adam, learning rate 2e-5, L1 physics loss,
10,000 updates, seed 0, full FP32 (about 16 minutes on an RTX 3090). Run with its defaults, the script writes
`weights/physics_trained_seed0.pth` and compares it tensor by tensor with `weights/physics_trained.pth`; on the
tested environment the refit is bit-identical. The number of updates and the seed were fixed without looking
at the ground truth: after 10,000 updates the physics loss is about 1e-3, the level reached by the original fit
(Physics_trained_net.ipynb). The fit depends on the random initialisation. With seeds 0, 1 and 2 the final PSNR
is 26.41 ± 1.70, 27.50 ± 1.63 and 25.55 ± 3.32 dB; Table 3 uses seed 0, the script default.

**Evaluation protocol (all notebooks).** The 10 field test patches (32 × 32 nodes, 400 m spacing) are
scaled to [-1, 1] per patch. PSNR, SSIM, relative error (RE) and RMSE are computed on the central 24 × 24
window and reported as mean ± standard deviation (ddof = 0) over the 10 patches. Networks run in full FP32
with deterministic cuDNN (set when `gravdc` is imported, see `gravdc/precision.py`).

In [1]:
import os
import sys

sys.path.insert(0, os.path.abspath(".."))  # make the gravdc package importable

import numpy as np
import torch

from gravdc import evaluate, format_summary, load_model, load_test_set, save_result, summarize
from gravdc.plotting import show_reconstruction
from gravdc.table3 import compare

DATA_DIR = "../data/test"
RESULTS_DIR = "../results"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| torch", torch.__version__)

device: cuda | torch 2.9.0+cu128


## Data
Observation: the ground-level field continued 800 m upward in a single step (zero padding to 36 × 36).

In [2]:
from gravdc import upward_continue

names, x = load_test_set(DATA_DIR)
x = x.to(device)
y = upward_continue(x, 800)
print(len(names), "patches:", names)

10 patches: ['data14.mat', 'data16.mat', 'data18.mat', 'data20.mat', 'data22.mat', 'data24.mat', 'data26.mat', 'data28.mat', 'data30.mat', 'data32.mat']


## Model and inference

In [3]:
model = load_model("physics_trained", device)
with torch.no_grad():
    x_hat = model(y)

## Metrics

In [4]:
per_image = evaluate(x, x_hat)
compare("Physics-Trained", summarize(per_image))
save_result("Physics-Trained", per_image, names, RESULTS_DIR)

Physics-Trained
  metric             this run              Table 3
  PSNR       26.4089 ± 1.7020     26.4089 ± 1.7020  ✓
  SSIM        0.8490 ± 0.1073      0.8490 ± 0.1073  ✓
  RE          0.2391 ± 0.0506      0.2391 ± 0.0506  ✓
  RMSE        0.0816 ± 0.0128      0.0816 ± 0.0128  ✓


In [5]:
# Figures of the field data are off by default (the executed notebooks are kept small and text-only).
# Set SHOW_FIGURE = True to look at a reconstruction.
SHOW_FIGURE = False
if SHOW_FIGURE:
    k = 0
    show_reconstruction(x[k], y[k], {"Physics-Trained": x_hat[k]}, title=names[k])